# LLM Document Summarizer — LLMOps with Amazon Bedrock

Building an LLM application is straightforward — call an API, return the result. Operating one in production is a different discipline. LLMOps asks: how do you know your prompt is working? How do you improve it without breaking what already works? How do you ensure the output is safe for all users? This project implements four LLMOps patterns on top of a document summarization system: **prompt versioning**, **automated evaluation**, **guardrails**, and **experiment tracking**.

In [ ]:
import json
from pathlib import Path
from config import (
    BEDROCK_REGION, BEDROCK_MODEL_ID, BEDROCK_GUARDRAIL_ID,
    ACTIVE_PROMPT_VERSION, PROMPTS_DIR, EVAL_DIR, EVAL_RESULTS_DIR,
    MLFLOW_EXPERIMENT_NAME, ROUGE_THRESHOLD, MAX_TEXT_LENGTH
)

print('=== Config ===')
print(f'Model:            {BEDROCK_MODEL_ID}')
print(f'Region:           {BEDROCK_REGION}')
print(f'Active prompt:    {ACTIVE_PROMPT_VERSION}')
print(f'MLflow exp:       {MLFLOW_EXPERIMENT_NAME}')
print(f'ROUGE threshold:  {ROUGE_THRESHOLD}')
print(f'Max text length:  {MAX_TEXT_LENGTH} chars')

## Why LLMOps Matters

A prompt is not code in the traditional sense — it has no type system, no unit tests, and no compiler. A small change to a prompt can dramatically improve or degrade output quality in ways that are hard to predict. Without a systematic approach to versioning and evaluation, prompt engineering is guesswork. LLMOps applies software engineering discipline to prompts: version control, regression testing, and performance tracking.

The cost of ignoring this is real. A prompt that works well on 10 documents may fail on the 11th. A prompt that was improved for one document type may regress on another. Without an evaluation pipeline, you have no way to know until a user complains. With one, you can quantify the tradeoff before you deploy.

## The Four LLMOps Patterns in This Project

### 1. Prompt Versioning

Prompts are stored as YAML files in `prompts/` with version numbers (v1, v2, v3). Switching the active version is a one-line config change in `config.py` — `ACTIVE_PROMPT_VERSION = 'v3'`. The full history of every prompt version is in Git, with creation date and a description of what changed. This is the prompt equivalent of semantic versioning for software: you always know what version is in production, what changed between versions, and you can roll back instantly.

### 2. Automated Evaluation

`evaluate.py` runs every prompt version against a held-out set of documents with human-written reference summaries and computes ROUGE scores — the standard NLP metric for summarization quality. Before deploying a new prompt version, run `evaluate.py`. If the new version scores below `ROUGE_THRESHOLD` on the eval set, it does not go to production. This is the same gate that prevents a model with degraded accuracy from being promoted in a classical ML system.

### 3. Bedrock Guardrails

Every Bedrock response passes through a guardrail before reaching the user. The guardrail anonymises PII (names, email addresses, phone numbers) and blocks harmful content. This means a document containing personal data can be summarised safely — the summary will not leak the individual's details. For enterprise document processing, this is a compliance requirement, not a nice-to-have. Every guardrail action is logged to MLflow so you can monitor input data quality over time.

### 4. MLflow Experiment Tracking

Every Bedrock call logs prompt version, latency, input/output length, and guardrails activity to MLflow. The eval pipeline logs ROUGE scores per prompt version. The MLflow UI shows the full history of prompt performance over time — exactly as it shows model training history in the credit-risk-scorer project. The same tool, the same mental model, applied to LLMs.

In [ ]:
# PDF extraction demo
# To run this cell, provide a PDF path. Using eval doc text as fallback.

from pdf_extractor import extract_text_from_pdf

sample_pdf = 'sample.pdf'  # Replace with your PDF path

if Path(sample_pdf).exists():
    result = extract_text_from_pdf(sample_pdf)
    print(f'Pages:     {result["page_count"]}')
    print(f'Words:     {result["word_count"]}')
    print(f'Truncated: {result["truncated"]}')
    print(f'Method:    {result["extraction_method"]}')
    print(f'\nFirst 500 chars:')
    print(result['text'][:500])
else:
    # Demonstrate with a plain text document instead
    sample_text = Path('eval/documents/doc1.txt').read_text()
    print('No PDF found — showing eval doc1 text:')
    print(f'Characters: {len(sample_text)}')
    print(f'Words:      {len(sample_text.split())}')
    print(f'\nFirst 500 chars:')
    print(sample_text[:500])

## Prompt Template Walkthrough

Three prompt versions are committed in `prompts/`. Each version builds on the previous:

| Version | Key addition | Intended improvement |
|---------|-------------|---------------------|
| v1 | Base structured summary | Establish baseline — summary, key_points, main_argument, sentiment, recommended_action |
| v2 | document_type + confidence | More specific classification; model signals how certain it is |
| v3 | risk_flags + target_audience | Extended for complex documents; surfaces concerns explicitly |

Switching versions is a one-line change in `config.py`. The prompt files are never modified — a new version means a new file.

In [ ]:
# Display all prompt versions
from prompt_manager import list_prompt_versions, load_prompt

versions = list_prompt_versions()
for v in versions:
    print(f"Version: {v['version']}")
    print(f"Created: {v['created']}")
    print(f"Description: {v['description']}")
    print()

In [ ]:
# Run the same document through all three prompt versions
# Requires AWS credentials and Bedrock access

from bedrock_client import summarize_document

doc_text = Path('eval/documents/doc1.txt').read_text()

results = {}
for version in ['v1', 'v2', 'v3']:
    print(f'Running {version}...')
    try:
        results[version] = summarize_document(doc_text, prompt_version=version)
        print(f'  Latency: {results[version]["latency_ms"]}ms')
        print(f'  Summary: {results[version]["summary"][:100]}...')
        if 'document_type' in results[version] and results[version]['document_type']:
            print(f'  Type: {results[version]["document_type"]}')
        if 'risk_flags' in results[version] and results[version]['risk_flags']:
            print(f'  Risk flags: {results[version]["risk_flags"]}')
    except Exception as e:
        print(f'  Error: {e}')
    print()

## Bedrock Guardrails

Guardrails are configured in the AWS console and applied via `invoke_model` with `guardrailIdentifier` and `guardrailVersion` parameters. When a guardrail fires, the action is returned in the HTTP response header `x-amzn-bedrock-guardrail-action`. When a guardrail fires:

- **ANONYMIZED**: PII detected and replaced with `[REDACTED]`. The summary is returned with a notice that anonymisation was applied. The document can still be processed — the user gets a useful summary without the leaked personal data.
- **BLOCKED**: Harmful content detected. A safe error response is returned instead of the model output. The user is told the content was blocked.

Guardrail activity is logged to MLflow so you can track how often documents trigger each action — useful for monitoring input data quality. A sudden spike in `guardrails_fired` could indicate a change in the document corpus (e.g. a new data source containing PII).

Guardrails run server-side inside Bedrock before the response is returned. This means PII redaction cannot be bypassed by a prompt injection attack — the redaction happens at the infrastructure layer, not the application layer.

In [ ]:
# Demonstrate guardrails by sending a document with fictional PII
# Requires AWS credentials and a configured Bedrock Guardrail

pii_document = """
Quarterly Performance Review — Employee: John Mitchell
Contact: john.mitchell@example.com | Phone: 555-123-4567
Address: 42 Oak Street, Portland, OR 97201

John Mitchell joined the engineering team in Q1 2025 and has demonstrated
strong performance in the infrastructure modernisation project. His team
delivered the Kubernetes migration two weeks ahead of schedule, reducing
infrastructure costs by 23%. Management recommends a salary review in Q3.
"""

try:
    result = summarize_document(pii_document, apply_guardrails=True)
    print('=== Guardrails result ===')
    print(f'Guardrails fired: {result["guardrails_fired"]}')
    print(f'Guardrails action: {result["guardrails_action"]}')
    print(f'Summary: {result["summary"]}')
    print()
    print('Note: PII (name, email, phone, address) should be anonymised above.')
except Exception as e:
    print(f'Error (check AWS credentials and guardrail ID): {e}')

## Evaluation Pipeline

### ROUGE Metrics

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) measures the overlap between a generated summary and a human-written reference:

- **ROUGE-1**: fraction of unigrams (single words) in the reference that appear in the generated summary
- **ROUGE-2**: fraction of bigrams (two-word sequences) that overlap
- **ROUGE-L**: longest common subsequence — measures whether the generated summary follows the same ordering of key phrases as the reference

**Intuition**: ROUGE-L measures the longest common subsequence between the generated summary and the reference summary — a higher score means more of the key phrases from the reference appear in the generated output. It is not a perfect metric — a summary can score low while still being good — but it provides a consistent, automated signal for comparing prompt versions systematically.

The eval set in `eval/` ships with five documents covering different document types (business report, research abstract, news, legal notice, technical README) with human-written reference summaries. The diversity ensures that a prompt version which over-fits to one document type does not artificially inflate the overall score.

In [ ]:
# Run the evaluation pipeline
# Requires AWS credentials and MLflow running (docker-compose up or mlflow server)
# This will take several minutes — it calls Bedrock once per (prompt version x document)

# Uncomment to run:
# import subprocess
# result = subprocess.run(['python', 'evaluate.py'], capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print('STDERR:', result.stderr)

# Or import and call directly:
# from evaluate import run_evaluation
# run_evaluation()

# Show most recent results if available
results_dir = Path('eval_results')
result_files = sorted(results_dir.glob('eval_*.json'), reverse=True) if results_dir.exists() else []

if result_files:
    with open(result_files[0]) as f:
        data = json.load(f)
    print(f'Most recent evaluation: {data["timestamp"]}')
    print(f'Best version: {data["best_version"]} (ROUGE-L: {data["best_rougeL"]:.3f})')
    print()
    print(f'{"Prompt":<8} {"ROUGE-1":<10} {"ROUGE-2":<10} {"ROUGE-L":<10} {"Latency"}')
    print('-' * 55)
    for row in data['results']:
        print(f'{row["version"]:<8} {row["rouge1"]:<10.3f} {row["rouge2"]:<10.3f} {row["rougeL"]:<10.3f} {row["latency_ms"]:.0f}ms')
else:
    print('No evaluation results yet.')
    print('Run: python evaluate.py (requires AWS credentials)')

## MLflow UI Walkthrough

With Docker Compose running, open the MLflow UI at **http://localhost:5000**. When running locally without Docker, run `mlflow ui --backend-store-uri mlruns` to view the local file-based tracking data.

The `llm-document-summarizer` experiment contains two types of runs:

1. **Production runs** — one run per `summarize_document()` call, named `summarize_v3_20260530_...`. Each run records:
   - `prompt_version`: which prompt was active
   - `latency_ms`: how long the Bedrock call took
   - `input_length` / `output_length`: token budget consumed
   - `guardrails_fired`: whether PII was detected or content blocked

2. **Evaluation runs** — one run per prompt version per `evaluate.py` execution, named `eval_v1`, `eval_v2`, `eval_v3`. Each run records:
   - `rouge1_avg`, `rouge2_avg`, `rougeL_avg`: average scores across the eval set
   - `latency_ms_avg`: average Bedrock latency
   - `guardrails_fired_count`: how many eval documents triggered guardrails

**Reading the comparison view**: in the MLflow Experiments tab, select all `eval_*` runs and click Compare. The parallel coordinates plot shows which prompt version dominates on ROUGE-L — the primary deployment criterion.

![MLflow UI — add screenshot after running evaluate.py]

This is the same pattern used in the credit-risk-scorer project, where MLflow tracks model training runs. The analogy is direct: prompt version = model checkpoint; ROUGE score = validation accuracy; guardrails activity = data quality flag.

## Connecting to the Broader Portfolio

This project extends the Bedrock work across four previous projects (IMDb sentiment, telco churn, spam classifier, stock trend) by adding the production management layer. The previous projects used Bedrock as a comparator — a zero-shot baseline to benchmark against trained models. This project uses Bedrock as the primary system, with the LLMOps infrastructure to manage, evaluate, and improve it over time.

The MLflow tracking pattern is consistent with credit-risk-scorer — the same tool is now used for both classical ML experiment tracking and LLM evaluation tracking. A team running both projects in production would have one observability system, one set of dashboards, and one operational process for monitoring model and prompt performance.

## Azure App Service Deployment

The API is deployed to Azure App Service using zip deployment. AWS credentials are set as Azure App Service Application Settings — they never appear in the codebase or Git history. This is the same pattern used in the stock-trend-sagemaker project.

```bash
# Create infrastructure
az group create --name llm-summarizer-rg --location westeurope
az appservice plan create --name llm-summarizer-plan \
  --resource-group llm-summarizer-rg --sku B1 --is-linux
# Scale to F1 via portal after creation

az webapp create --name llm-document-summarizer \
  --resource-group llm-summarizer-rg \
  --plan llm-summarizer-plan \
  --runtime "PYTHON:3.11"

# Set startup command
az webapp config set --name llm-document-summarizer \
  --resource-group llm-summarizer-rg \
  --startup-file "gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600"

# Set credentials as app settings — never in code
az webapp config appsettings set --name llm-document-summarizer \
  --resource-group llm-summarizer-rg \
  --settings \
    SCM_DO_BUILD_DURING_DEPLOYMENT=true \
    AWS_ACCESS_KEY_ID=<your-key> \
    AWS_SECRET_ACCESS_KEY=<your-secret> \
    AWS_DEFAULT_REGION=us-east-1 \
    BEDROCK_GUARDRAIL_ID=<your-guardrail-id>

# Package and deploy
zip -r deploy.zip . -x "*.git*" -x "venv/*" -x "__pycache__/*" \
  -x "*.ipynb_checkpoints*" -x "mlruns/*" -x "uploads/*"
az webapp deployment source config-zip \
  --name llm-document-summarizer \
  --resource-group llm-summarizer-rg \
  --src deploy.zip
```

Note: MLflow is not deployed to Azure — on the deployed instance, logging calls fail silently and all other functionality works. For production MLflow tracking, deploy a dedicated MLflow server or use a managed tracking service.

## Key Findings

Evaluated three prompt versions across five held-out documents (business report, research abstract, news article, legal notice, technical README) using Claude Haiku via Amazon Bedrock.

| Prompt | ROUGE-1 | ROUGE-2 | ROUGE-L | Avg Latency |
|--------|---------|---------|---------|-------------|
| v1     | 0.595   | 0.352   | 0.497   | 3857ms      |
| v2     | 0.532   | 0.249   | 0.379   | 4170ms      |
| v3     | 0.609   | 0.350   | 0.504   | 5105ms      |

**Winner: v3** (ROUGE-L 0.504) — the extended prompt with `risk_flags` and `target_audience` fields consistently outperformed v1 and v2 across all five document types.

**v2 underperformed v1** despite being more detailed. The additional `document_type` and `confidence` fields appear to consume output tokens that would otherwise contribute to the summary content, reducing lexical overlap with the reference summaries.

**Latency tradeoff**: v3 adds ~1.2s over v1 (5105ms vs 3857ms) due to the longer prompt and extended output schema. Acceptable for a document summarization use case where quality matters more than sub-second latency.

**Guardrails firing rate**: 0/15 calls on the eval set — none of the five evaluation documents contain PII or harmful content, as expected.

**Active prompt updated to v3** in `config.py`.

## What LLMOps Adds Beyond a Basic LLM Wrapper

| Concern | Basic LLM wrapper | This project |
|---------|------------------|--------------|
| Prompt management | Hardcoded string in application code | Versioned YAML files in `prompts/` with Git history |
| Quality assurance | None — output is whatever the model returns | ROUGE evaluation on held-out set before promotion |
| Safety | None — raw model output returned to user | Bedrock Guardrails: PII anonymised, harmful content blocked |
| Observability | None | MLflow tracking per call: latency, prompt version, guardrails |
| Rollback | Re-deploy with previous code | Change one config line: `ACTIVE_PROMPT_VERSION = 'v1'` |
| Improvement process | Guesswork — change the prompt and hope | Evaluate → compare ROUGE scores → promote if above threshold |

The table shows the pattern: an LLM wrapper solves the inference problem. An LLMOps system solves the operational problems that arise when inference runs in production at scale.